# IRLC Math Helper Notebook

Small symbolic and numeric helpers for IRLC exam practice.

Use this notebook for quick checks of:

- derivatives, gradients, Jacobians, Hessians,
- Taylor expansions and linearizations,
- Euler discretization,
- symbolic/numeric integration,
- simple expectations,
- Bellman-style expected costs/values,
- one-step LQR feedback gains.

The helpers are meant for checking work and speeding up practice. For written exam answers, still show the relevant formula and reasoning.

## Imports

Run this first. Everything below only uses packages already listed in the project dependencies.

In [1]:
import numpy as np
import sympy as sp
from scipy.integrate import quad

sp.init_printing()

## Core Symbolic Helpers

These wrappers keep the most common operations short and consistent.

In [2]:
def col(xs):
    """Return xs as a SymPy column vector."""
    return sp.Matrix(xs)


def simplify(x):
    """Simplify expressions, matrices, lists, or tuples."""
    if isinstance(x, sp.MatrixBase):
        return x.applyfunc(sp.simplify)
    if isinstance(x, (list, tuple)):
        return type(x)(sp.simplify(v) for v in x)
    return sp.simplify(x)


def gradient(f, xs):
    """Gradient of scalar f with respect to variables xs as a column vector."""
    return simplify(col([sp.diff(f, x) for x in xs]))


def jacobian(fs, xs):
    """Jacobian of vector expression fs with respect to variables xs."""
    return simplify(col(fs).jacobian(xs))


def hessian(f, xs):
    """Hessian of scalar f with respect to variables xs."""
    return simplify(sp.hessian(f, xs))


def subs_eval(expr, values):
    """Substitute values into an expression or matrix and simplify."""
    return simplify(expr.subs(values))


def numeric(expr, values=None):
    """Evaluate a SymPy expression/matrix numerically."""
    if values:
        expr = expr.subs(values)
    return np.array(expr, dtype=float) if isinstance(expr, sp.MatrixBase) else float(expr)

## Differentiation Examples

In [ ]:
x, y, u = sp.symbols('x y u', real=True)

f = sp.exp(-x) * u + x**2 * y
print('df/dx =')
display(sp.diff(f, x))

print('gradient wrt [x, y] =')
display(gradient(f, [x, y]))

print('hessian wrt [x, y] =')
display(hessian(f, [x, y]))

## Jacobians and Linearization

For dynamics

$$
x_{k+1} = f(x_k, u_k),
$$

the first-order linearization around $(\bar{x}, \bar{u})$ is

$$
x_{k+1} \approx f(\bar{x},\bar{u}) + A(x_k-\bar{x}) + B(u_k-\bar{u}).
$$

Equivalently:

$$
x_{k+1} \approx A x_k + B u_k + d,
$$

where

$$
d = f(\bar{x},\bar{u}) - A\bar{x} - B\bar{u}.
$$

In [ ]:
def linearize(f_vec, x_vars, u_vars, x_bar, u_bar):
    """
    Linearize vector dynamics f_vec around (x_bar, u_bar).

    Returns:
        A, B, d, f0
    so that f(x,u) approx A*x + B*u + d.
    """
    f_vec = col(f_vec)
    x_vec = col(x_vars)
    u_vec = col(u_vars)

    subs = {var: val for var, val in zip(x_vars, x_bar)}
    subs.update({var: val for var, val in zip(u_vars, u_bar)})

    A = jacobian(f_vec, x_vars).subs(subs)
    B = jacobian(f_vec, u_vars).subs(subs)
    f0 = f_vec.subs(subs)
    d = f0 - A * col(x_bar) - B * col(u_bar)
    return simplify(A), simplify(B), simplify(d), simplify(f0)

In [ ]:
# Example: nonlinear discrete dynamics
x1, x2, u = sp.symbols('x1 x2 u', real=True)
f_vec = [x1 + x2, sp.cos(x1 + u)]

A, B, d, f0 = linearize(
    f_vec=f_vec,
    x_vars=[x1, x2],
    u_vars=[u],
    x_bar=[0, 0],
    u_bar=[0],
)

print('A =')
display(A)
print('B =')
display(B)
print('d =')
display(d)
print('f0 =')
display(f0)

## Euler Discretization

For continuous dynamics

$$
\dot{x} = f(x,u),
$$

Euler discretization with step size $\Delta$ gives

$$
x_{k+1} = x_k + \Delta f(x_k,u_k).
$$

In [ ]:
def euler_discretize(f_cont, x_vars, dt):
    """Return Euler-discretized dynamics x_next = x + dt*f_cont."""
    return simplify(col(x_vars) + dt * col(f_cont))


Delta = sp.symbols('Delta', positive=True)
w, v, u = sp.symbols('w v u', real=True)

# Example pendulum-like dynamics: w_dot = v, v_dot = cos(w + u)
f_cont = [v, sp.cos(w + u)]
f_disc = euler_discretize(f_cont, [w, v], Delta)

display(f_disc)

## Taylor Expansions

In [ ]:
def taylor_1d(f, var, around=0, order=2):
    """1D Taylor expansion through the given order."""
    return sp.series(f, var, around, order + 1).removeO().simplify()


def first_order_taylor(f, xs, point):
    """First-order multivariate Taylor approximation of scalar f around point."""
    values = dict(zip(xs, point))
    result = f.subs(values)
    for x_i, p_i in zip(xs, point):
        result += sp.diff(f, x_i).subs(values) * (x_i - p_i)
    return sp.simplify(result)


z = sp.symbols('z')
display(taylor_1d(sp.sin(z), z, around=0, order=5))

g = sp.exp(x + y)
display(first_order_taylor(g, [x, y], [0, 0]))

## Integration Helpers

In [ ]:
def integrate_symbolic(f, var, lower=None, upper=None):
    """Symbolic indefinite or definite integral."""
    if lower is None or upper is None:
        return sp.integrate(f, var)
    return sp.integrate(f, (var, lower, upper))


def integrate_numeric(func, lower, upper):
    """Numeric definite integral. func should be a Python function."""
    value, error = quad(func, lower, upper)
    return value, error


t = sp.symbols('t', real=True)
expr = t**2 + sp.E

print('symbolic integral 0 to T:')
T = sp.symbols('T', positive=True)
display(integrate_symbolic(expr, t, 0, T))

print('numeric integral 0 to 1:')
print(integrate_numeric(lambda tau: tau**2 + np.e, 0, 1))

## Expectations

For discrete outcomes:

$$
E[f(X)] = \sum_x f(x)p(x).
$$

This comes up constantly in DP, MDPs, inventory control, and Bellman equations.

In [ ]:
def expectation_discrete(outcomes, probs=None, func=lambda x: x):
    """
    Compute E[func(X)] for finite outcomes.

    outcomes can be:
      - list of values, with probs list supplied
      - dict value -> probability
    """
    if isinstance(outcomes, dict):
        items = list(outcomes.items())
    else:
        if probs is None:
            raise ValueError('probs must be supplied when outcomes is not a dict')
        items = list(zip(outcomes, probs))
    return simplify(sum(sp.Rational(1, 1) * p * func(x) for x, p in items))


# Example: X in {0, 1, 2}, probabilities {1/4, 1/2, 1/4}
D = {0: sp.Rational(1, 4), 1: sp.Rational(1, 2), 2: sp.Rational(1, 4)}
print('E[D] =')
display(expectation_discrete(D))
print('E[D^2] =')
display(expectation_discrete(D, func=lambda d: d**2))

## Bellman / DP Expected Cost Helper

For finite-horizon cost minimization:

$$
J_k(x) = \min_u E[g_k(x,u,w) + J_{k+1}(f_k(x,u,w))].
$$

In [ ]:
def expected_dp_q(noise_probs, stage_cost, next_value, transition):
    """
    Compute Q(x,u) = E_w[stage_cost(x,u,w) + next_value(transition(x,u,w))].

    noise_probs: dict w -> probability
    stage_cost, next_value, transition: functions using symbolic variables
    """
    return simplify(sum(
        p * (stage_cost(w) + next_value(transition(w)))
        for w, p in noise_probs.items()
    ))


# Example from a final-stage DP style question
x, u, sigma = sp.symbols('x u sigma', real=True, positive=True)
# For standard normal w, use E[w]=0, E[w^2]=1 by expanding manually:
w = sp.symbols('w', real=True)
expr = (x - u + w)**2
E_expr = sp.expand(expr).subs({w: 0}) + 1  # quick pattern for E[w]=0, E[w^2]=1 does not handle all powers
print('For (x-u+w)^2 with w~N(0,1), use:')
display((x-u)**2 + 1)

## MDP Bellman Helpers

For reward maximization:

$$
v_\pi(s) = \sum_a \pi(a|s) \sum_{s',r}p(s',r|s,a)[r + \gamma v_\pi(s')]
$$

and

$$
v_*(s) = \max_a \sum_{s',r}p(s',r|s,a)[r + \gamma v_*(s')].
$$

In [ ]:
def bellman_action_value(transitions, gamma, V):
    """
    Expected one-step lookahead for one action.

    transitions: list of (probability, next_state, reward)
    V: dict next_state -> value
    """
    return simplify(sum(p * (r + gamma * V[s_next]) for p, s_next, r in transitions))


gamma = sp.symbols('gamma', real=True)
V = {'s1': sp.symbols('v1'), 's2': sp.symbols('v2')}
transitions = [
    (sp.Rational(1, 2), 's1', 1),
    (sp.Rational(1, 2), 's2', 0),
]
display(bellman_action_value(transitions, gamma, V))

## One-Step LQR Gain

For linear dynamics

$$
x_{k+1} = A x_k + B u_k
$$

and quadratic cost-to-go with next value matrix $V_{k+1}$, the feedback gain is often of the form

$$
L_k = -(R + B^T V_{k+1} B)^{-1} B^T V_{k+1} A.
$$

Use this to check scalar or small-matrix calculations.

In [ ]:
def lqr_feedback_gain(A, B, R, V_next):
    """Return L = -(R + B.T V B)^(-1) B.T V A."""
    A = sp.Matrix(A)
    B = sp.Matrix(B)
    R = sp.Matrix(R)
    V_next = sp.Matrix(V_next)
    return simplify(-(R + B.T * V_next * B).inv() * B.T * V_next * A)


a = sp.symbols('a', real=True)
A = sp.Matrix([[a]])
B = sp.Matrix([[1]])
R = sp.Matrix([[2]])
V_next = sp.Matrix([[2]])

display(lqr_feedback_gain(A, B, R, V_next))

## Numeric Lambdify Helper

Turn symbolic formulas into functions for quick checks.

In [ ]:
def make_numeric(expr, vars):
    """Create a NumPy-compatible numeric function from a SymPy expression."""
    return sp.lambdify(vars, expr, modules='numpy')


expr = sp.log(t**2 + sp.E)
fn = make_numeric(expr, [t])
print(fn(1.0))

## Quick Reference Cells

Copy and edit these patterns during practice.

In [ ]:
# Symbols
x1, x2, u, Delta, gamma, alpha = sp.symbols('x1 x2 u Delta gamma alpha', real=True)

# Vector dynamics template
f_vec = [x1 + Delta*x2, x2 + Delta*sp.cos(x1 + u)]
A, B, d, f0 = linearize(f_vec, [x1, x2], [u], x_bar=[0, 0], u_bar=[0])

# TD/Q update templates
R, Qsa, Qnext = sp.symbols('R Qsa Qnext')
td_update = Qsa + alpha * (R + gamma * Qnext - Qsa)

print('A, B, d for template dynamics:')
display(A, B, d)
print('TD-style update:')
display(td_update)